In [1]:
import json
import sys
from pathlib import Path

# 路径设置
sys.path.append(str(Path.cwd()))

from config import (
    NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD,
    ENTITY_LABELS, RELATION_TYPES, SIMILARITY_THRESHOLD
)
from kg_client import Neo4jClient
from utils import ollama_embed, cosine_similarity, triple_to_text

print("配置加载完成。")

配置加载完成。


In [2]:
# 单元格 2：初始化 Neo4j 客户端并测试连接
client = Neo4jClient()

# 简单测试查询
try:
    result = client.query("MATCH (n) RETURN count(n) AS total")
    total_nodes = result[0]['total'] if result else 0
    print(f"Neo4j 连接成功！图谱中共有 {total_nodes} 个节点。")
except Exception as e:
    print(f"Neo4j 连接失败：{e}")
    client = None

Neo4j 连接成功！图谱中共有 1212 个节点。


In [3]:
# 单元格 3：加载第二步提取的实体 Key
keys_file = "data/extracted_keys.json"
if Path(keys_file).exists():
    with open(keys_file, "r", encoding="utf-8") as f:
        keys = json.load(f)
    print(f"已加载提取的实体 Key，共 {len(keys.get('entities', []))} 个实体。")
else:
    print("未找到提取文件，使用测试实体。")
    keys = {
        "entities": ["华佗五禽戏", "五种动物", "传承人"]
    }

已加载提取的实体 Key，共 3 个实体。


In [4]:
# 单元格 4：定义检索参数与函数
# ========== 可调参数 ==========
MAX_HOPS = 2          # 检索的最大跳数
TOP_K_NODES = 3       # 每个实体匹配后取前几个节点进行扩展
# =============================

def retrieve_by_entity(entity_name, client, fuzzy=True, max_hops=MAX_HOPS, top_k_nodes=TOP_K_NODES):
    """检索实体的多跳邻居三元组"""
    nodes = client.get_node_by_name(entity_name, fuzzy=fuzzy)
    if not nodes:
        return []
    
    all_triples = set()
    for node in nodes[:top_k_nodes]:
        node_id = node.get("element_id")
        if not node_id:
            continue
        
        if max_hops == 1:
            cypher = """
                MATCH (n)-[r]->(m)
                WHERE elementId(n) = $nid
                RETURN n.name AS start, type(r) AS relation, m.name AS end
                UNION
                MATCH (n)<-[r]-(m)
                WHERE elementId(n) = $nid
                RETURN m.name AS start, type(r) AS relation, n.name AS end
            """
        else:
            cypher = f"""
                MATCH path = (start)-[r*1..{max_hops}]-(end)
                WHERE elementId(start) = $nid
                UNWIND relationships(path) AS rel
                WITH startNode(rel) AS s, rel, endNode(rel) AS e
                RETURN s.name AS start, type(rel) AS relation, e.name AS end
            """
        results = client.query(cypher, {"nid": node_id})
        for rec in results:
            if rec["start"] and rec["relation"] and rec["end"]:
                all_triples.add((rec["start"], rec["relation"], rec["end"]))
    return list(all_triples)

def semantic_filter(candidate_triples, key_texts, threshold=SIMILARITY_THRESHOLD):
    """基于嵌入相似度过滤三元组"""
    if not candidate_triples:
        return []
    
    triple_texts = [triple_to_text(t) for t in candidate_triples]
    print("  正在计算 Key 文本嵌入...")
    key_embeddings = [ollama_embed(text) for text in key_texts]
    
    filtered = []
    for triple, text in zip(candidate_triples, triple_texts):
        triple_emb = ollama_embed(text)
        max_sim = max(cosine_similarity(triple_emb, k_emb) for k_emb in key_embeddings)
        if max_sim >= threshold:
            filtered.append(triple)
    return filtered

In [5]:
# 单元格 5：执行实体检索并收集候选三元组
all_candidate_triples = set()

for entity in keys.get("entities", []):
    triples = retrieve_by_entity(entity, client, fuzzy=True)
    print(f"实体 '{entity}' 检索到 {len(triples)} 条候选三元组")
    all_candidate_triples.update(triples)

print(f"\n去重后候选三元组总数：{len(all_candidate_triples)}")

      [Neo4j] 查找 '陈氏太极拳' (fuzzy=True) 返回 10 个节点
实体 '陈氏太极拳' 检索到 52 条候选三元组
      [Neo4j] 查找 '动作要素' (fuzzy=True) 返回 0 个节点
实体 '动作要素' 检索到 0 条候选三元组
      [Neo4j] 查找 '传承人' (fuzzy=True) 返回 2 个节点
实体 '传承人' 检索到 132 条候选三元组

去重后候选三元组总数：182


In [6]:
# 单元格 6：语义相似度过滤
print("\n开始语义相似度过滤...")

# 构建用于相似度比较的 Key 文本集合（仅实体名称）
key_texts = keys["entities"]
print(f"共 {len(key_texts)} 个实体文本用于语义过滤")

filtered_triples = semantic_filter(list(all_candidate_triples), key_texts, threshold=SIMILARITY_THRESHOLD)
print(f"过滤后保留三元组数量：{len(filtered_triples)}")


开始语义相似度过滤...
共 3 个实体文本用于语义过滤
  正在计算 Key 文本嵌入...
过滤后保留三元组数量：168


In [7]:
# 单元格 7：展示部分过滤结果
print("\n===== 过滤后的三元组示例（前20条）=====")
for i, (s, r, o) in enumerate(filtered_triples[:20], 1):
    print(f"{i}. ({s}) -[{r}]-> ({o})")


===== 过滤后的三元组示例（前20条）=====
1. (孙氏太极拳) -[传承]-> (太极拳)
2. (周恩来) -[传承]-> (太极拳)
3. (太极拳) -[参与]-> (世界百城千万人太极拳集中展演活动)
4. (吴氏太极拳) -[传承]-> (太极拳)
5. (王雁) -[传承]-> (陈氏太极拳)
6. (太极拳) -[参与]-> (中国太极功夫文化旅游)
7. (太极拳) -[展现]-> (太极)
8. (太极拳) -[参与]-> (第十届国际太极拳交流大赛)
9. (张全亮) -[传承]-> (陈氏太极拳)
10. (太极拳) -[传承]-> (无用)
11. (太极拳) -[展现]-> (平和、包容、友善)
12. (陈氏) -[传承]-> (陈氏太极拳)
13. (陈照丕) -[传承]-> (陈氏太极拳)
14. (太极拳) -[创编]-> (功法)
15. (中医导引法) -[传承]-> (太极拳)
16. (太极拳) -[传承]-> (吴氏太极拳)
17. (陈氏太极拳) -[参与]-> (太极拳（陈氏太极拳）)
18. (太极拳) -[展现]-> (动静结合)
19. (王西安) -[创编]-> (陈氏太极拳推手技法)
20. (太极拳) -[传承]-> (家族传承)


In [8]:
# 单元格 8：保存结果供下一步使用
output_file = "data/retrieved_triples_local.json"
Path("data").mkdir(exist_ok=True)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(filtered_triples, f, ensure_ascii=False, indent=2)

print(f"本地层面检索结果已保存至：{output_file}")

本地层面检索结果已保存至：data/retrieved_triples_local.json
